# [WIP] Nemotron-3-Nano-30B LoRA for Model Reasoning Challenge

This notebook is a prize-eligibility writeup skeleton for the NVIDIA Nemotron Model Reasoning Challenge on Kaggle. The competition uses Nemotron-3-Nano-30B as the base model and expects a LoRA adapter submission, while prize eligibility also requires a public notebook and write-up.[web:1]

## 1. Overview

Use this section to explain the goal of the solution, the core approach, and what parts were trained off-Kaggle versus demonstrated inside this notebook.[web:1]

Suggested bullets:
- Competition objective.
- Final submission artifact.
- High-level method: SFT, synthetic data, DSPy usage, PEFT/LoRA choices.

## 2. Environment and Dependencies

The notebook can document the runtime dependencies even if the full training run happened on external hardware.[web:1]

In [ ]:
!pip install -q transformers==4.57.3 peft==0.14.0 accelerate trl datasets

## 3. Loading Base Model and LoRA Adapter

This section shows how to load the Nemotron base model together with the final adapter. If the adapter is published to Hugging Face Hub, loading it from a model repo is a clean way to demonstrate reuse in the public notebook.[web:160]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
ADAPTER_REPO = "your-username/nemotron-nano-30b-lora-kaggle"
USE_HF_ADAPTER = True


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

if USE_HF_ADAPTER:
    model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
else:
    ADAPTER_PATH = "/kaggle/input/your-adapter-dataset/adapter"
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()

## 4. Prompt Template and Inference Demo

The competition metric prefers answers inside `\boxed{}` when present, so the demonstration prompt should use that final-answer format.[web:1]

In [ ]:
def format_messages(problem: str):
    return [
        {
            "role": "system",
            "content": (
                "You are a careful reasoning model. Solve the problem step by step, "
                "and end with a single line of the form Final answer: \boxed{...}."
            ),
        },
        {"role": "user", "content": problem},
    ]


def generate_answer(problem: str, max_new_tokens: int = 256):
    messages = format_messages(problem)
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = "
".join(f"{m['role']}: {m['content']}" for m in messages) + "
assistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("What is 17 + 25?"))

## 5. Data Pipeline

Describe the public datasets, any filtering, any synthetic data generation, and how examples were converted into JSONL rows with `prompt`, `response`, and optional `system` fields for supervised fine-tuning.[web:107][web:110]

In [ ]:
example_row = {
    "prompt": "What is 17 + 25?",
    "response": "17 + 25 = 42.\nFinal answer: \boxed{42}",
    "system": "You are a careful reasoning model. Solve the problem step by step and end with Final answer: \boxed{...}."
}
example_row

## 6. Training Configuration and Loop

The full training run can be described here even if it was executed on GB10 hardware outside Kaggle. TRL's `SFTTrainer` supports the text-column supervised fine-tuning pattern used in the provided training script.[web:107]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import LoraConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

train_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
)

# trainer = SFTTrainer(
#     model=base_model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=valid_dataset,
#     peft_config=peft_config,
#     args=train_args,
#     dataset_text_field="text",
#     max_seq_length=4096,
# )
# trainer.train()

## 7. Evaluation

Use this section to explain local validation. A good lightweight approximation is to extract the final `\boxed{}` answer and compare it with numeric tolerance when needed.[web:1]

In [ ]:
import math
import re

BOXED_RE = re.compile(r"\\boxed\{([^{}]+)\}")


def extract_boxed(text: str) -> str:
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else ""


def compare_numeric(pred: str, truth: str, rel_tol: float = 1e-4) -> bool:
    try:
        return math.isclose(float(pred), float(truth), rel_tol=rel_tol)
    except Exception:
        return pred.strip() == truth.strip()

## 8. Results and Discussion

Summarize validation results, leaderboard results, ablations, and the most important lessons from the iteration process.[web:1]

## 9. Reproducibility Notes

List the exact base model, LoRA configuration, training scripts, Docker environment, and where the final adapter is hosted. If the adapter is uploaded to Hugging Face Hub, include the repo ID here.[web:160]

## 10. Acknowledgements

Credit NVIDIA, Kaggle, Hugging Face PEFT/TRL, DSPy, and any public datasets used in the work.[web:1][web:35][web:89]